# FabricPC Transformer V2

Trains the decomposed PC Transformer model on the Tiny Shakespeare dataset,
evaluates its performance, and generates sample text using temperature sampling.

**Architecture** (each transformer block decomposed into separate PC nodes):

```
input ──→ Embedding ──→ MhaResidual_0 ──→ LnMlp1_0 ──→ Mlp2Residual_0 ──→ ... ──→ VocabProjection
                              ↑                              ↑
                         mask │                       (skip from MhaResidual)
```

Each block: `MhaResidual(in + mask) ──→ LnMlp1 ──→ Mlp2Residual(in + skip)`

Supports two training modes:
- **Predictive Coding (PC):** Local Hebbian learning with multi-GPU pmap support
- **Backpropagation:** Standard end-to-end gradient training (single device)

**Results** (default call, cuda12, rtx3090, jax 0.8.1):
```
Model parameters: 108,353
Vocab Size: 65
Train Epoch 1/5, Energy: 274.3637, Loss: 2.1401, Perplexity: 8.50
Train Epoch 2/5, Energy: 260.0219, Loss: 2.0280, Perplexity: 7.60
Train Epoch 3/5, Energy: 250.6280, Loss: 1.9546, Perplexity: 7.06
Train Epoch 4/5, Energy: 244.7030, Loss: 1.9089, Perplexity: 6.75
Train Epoch 5/5, Energy: 242.0046, Loss: 1.8878, Perplexity: 6.61
Training completed in 8772.4s
Evaluation completed in 55.2s
Test Accuracy:   35.92%
Test CE Loss:    2.2108
Test Perplexity: 9.12
--- Generating ---
ROMEO: whou sarone the bro beariers thas tray sucas a st my lo the to ate.
```

## Imports & Setup

In [1]:
import jax
import jax.numpy as jnp
from fabricpc.graph_initialization import initialize_params
from fabricpc.training import (
    train_autoregressive,
    evaluate_autoregressive,
    train_backprop_autoregressive,
    evaluate_backprop_autoregressive,
    generate_autoregressive,
)
from fabricpc.core.inference import InferenceSGDNormClip
from fabricpc.models import create_deep_transformer
from fabricpc.utils.data import CharDataLoader, BpeDataLoader
import optax
import time
from fabricpc import setup_jax

setup_jax()

## Tuned Defaults

In [2]:
BPE_DEFAULTS = {
    "embed_dim": 128,
    "num_heads": 4,
    "mlp_dim": 512,
    "depth": 4,
    "seq_len": 64,
    "batch_size": 32,
    "num_epochs": 5,
    "infer_steps": 23,
    "lr": 1.676456563307537e-05,
    "eta_infer": 0.06558512264378524,
    "weight_init_std": 0.039890499730518045,
}

CHAR_DEFAULTS = {
    "embed_dim": 64,
    "num_heads": 8,
    "mlp_dim": 256,
    "depth": 2,
    "seq_len": 128,
    "batch_size": 16,
    "num_epochs": 5,
    "infer_steps": 12,
    "lr": 0.00012108621644524519,
    "eta_infer": 0.0174852165627398,
    "weight_init_std": 0.015166293102182283,
}

## Configuration

Set `tokenizer = 'bpe'` to switch to BPE tokenization.

In [3]:
# Configuration (replaces argparse) — uses CHAR_DEFAULTS
mode = "pc"          # "pc" or "backprop"
tokenizer = "char"   # "char" or "bpe"
num_epochs = 5
seed = 42
verbose = False

# Use char defaults (override individual values as needed)
embed_dim = CHAR_DEFAULTS["embed_dim"]
num_heads = CHAR_DEFAULTS["num_heads"]
mlp_dim = CHAR_DEFAULTS["mlp_dim"]
depth = CHAR_DEFAULTS["depth"]
seq_len = CHAR_DEFAULTS["seq_len"]
batch_size = CHAR_DEFAULTS["batch_size"]
infer_steps = CHAR_DEFAULTS["infer_steps"]
lr = CHAR_DEFAULTS["lr"]
eta_infer = CHAR_DEFAULTS["eta_infer"]
weight_init_std = CHAR_DEFAULTS["weight_init_std"]

## Load Data

In [ ]:
use_pc = mode == "pc"
use_bpe = tokenizer == "bpe"
print(f"JAX devices: {jax.devices()}")
# Batch size
if use_pc:
    n_devices = jax.device_count()
    batch_size = batch_size * n_devices
    print(f"PC mode: {n_devices} device(s), total batch_size={batch_size}")
else:
    print(f"Backprop mode: single device, batch_size={batch_size}")

# Data
if use_bpe:
    train_loader = BpeDataLoader(
        "train", seq_len=seq_len, batch_size=batch_size, shuffle=True, seed=seed,
    )
    test_loader = BpeDataLoader(
        "test", seq_len=seq_len, batch_size=batch_size, shuffle=False
    )
else:
    train_loader = CharDataLoader(
        "train", seq_len=seq_len, batch_size=batch_size, shuffle=True, seed=seed,
    )
    test_loader = CharDataLoader(
        "test", seq_len=seq_len, batch_size=batch_size, shuffle=False
    )

vocab_size = train_loader.vocab_size
char_to_ix = train_loader.token_to_idx if use_bpe else train_loader.char_to_idx

PC mode: 1 device(s), total batch_size=16


## Build Model

In [5]:
structure = create_deep_transformer(
    depth=depth,
    embed_dim=embed_dim,
    num_heads=num_heads,
    mlp_dim=mlp_dim,
    seq_len=seq_len,
    vocab_size=vocab_size,
    inference=InferenceSGDNormClip(
        eta_infer=eta_infer,
        infer_steps=infer_steps,
        max_norm=5.0,
        latent_decay=0.0,
    ),
    weight_init={"type": "normal", "std": weight_init_std},
)

master_rng_key = jax.random.PRNGKey(seed)
graph_key, train_key, eval_key = jax.random.split(master_rng_key, 3)

params = initialize_params(structure, graph_key)
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Model parameters: {n_params:,}")

Model parameters: 108,353


## Train

In [6]:
train_config = {
    "num_epochs": num_epochs,
    "use_causal_mask": True,
}
steps_per_epoch = train_loader.num_sequences // batch_size
schedule = optax.cosine_decay_schedule(
    init_value=lr,
    decay_steps=num_epochs * steps_per_epoch,
    alpha=0.1,
)
optimizer = optax.adam(schedule)

print(f"Vocab Size: {vocab_size}")
start = time.time()

def iter_callback(epoch_idx, batch_idx, energy):
    if verbose and (batch_idx + 1) % 50 == 0:
        print(
            f"Epoch {epoch_idx + 1} | Batch {batch_idx + 1} | Energy: {energy:.4f}"
        )
    return energy

if use_pc:
    trained_params, _, _ = train_autoregressive(
        params, structure, train_loader, optimizer, train_config, train_key,
        verbose=True, iter_callback=iter_callback,
    )
else:
    trained_params, _, _ = train_backprop_autoregressive(
        params, structure, train_loader, optimizer, train_config, train_key, verbose=True,
    )

print(f"Training completed in {time.time() - start:.1f}s")

Vocab Size: 65


KeyboardInterrupt: 

## Evaluate

In [ ]:
eval_start = time.time()
if use_pc:
    metrics = evaluate_autoregressive(
        trained_params, structure, test_loader, train_config, eval_key
    )
else:
    metrics = evaluate_backprop_autoregressive(
        trained_params, structure, test_loader, train_config, eval_key
    )
print(f"Evaluation completed in {time.time() - eval_start:.1f}s")

print(f"Test Accuracy:   {metrics['accuracy'] * 100:.2f}%")
print(f"Test CE Loss:    {metrics['loss']:.4f}")
print(f"Test Perplexity: {metrics['perplexity']:.2f}")

## Generate Text

In [ ]:
gen_key = jax.random.PRNGKey(99)
prompt_text = "ROMEO: "
if use_bpe:
    seed_indices = train_loader._tok.encode(prompt_text).ids
else:
    seed_indices = [char_to_ix.get(c, 0) for c in prompt_text]

current_indices = ([0] * (seq_len - len(seed_indices)) + seed_indices)[-seq_len:]
prompt = jnp.array(current_indices, dtype=jnp.int32)

print("--- Generating ---")
generated = generate_autoregressive(
    trained_params, structure, prompt, max_new_tokens=200, rng_key=gen_key, temperature=0.8,
)

generated_ids = generated[seq_len:]
print(prompt_text + train_loader.decode(generated_ids))